In [1]:
import pandas as pd
import numpy as np

In [2]:
import pandas as pd
from sqlalchemy import create_engine


USER = ""
PASSWORD = ""
HOST = ""
PORT = ""
DATABASE = ""


DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"


engine = create_engine(DATABASE_URL)


df = pd.read_sql("SELECT * FROM dim_customers LIMIT 5", engine)

print(df)

   customer_id     customer_name            city currency
0       789101  Costco Wholesale  New Jersey, US      USD
1       789102      Vijay Stores       Ahmedabad      INR
2       789103      Vijay Stores        Vadodara      INR
3       789121          Coolblue       Ahmedabad      INR
4       789122          Coolblue        Vadodara      INR


In [3]:
dim_customers_query=""" 
select * from dim_customers
"""
dim_products_query=""" 
select * from dim_products
"""
dim_target_orders_query=""" 
select * from dim_targets_orders
"""
fact_aggregate_query=""" 
select * from fact_aggregate
"""
fact_order_line_query=""" 
select * from fact_order_line
"""


dim_customers = pd.read_sql(dim_customers_query,engine)
dim_products = pd.read_sql(dim_products_query,engine)
dim_target_orders = pd.read_sql(dim_target_orders_query,engine)
fact_aggregate = pd.read_sql(fact_aggregate_query,engine)
fact_order_line = pd.read_sql(fact_order_line_query,engine)

In [4]:
fact_aggregate

,order_id,customer_id,order_placement_date,on_time,in_full,otif
0,FMR32103503,789103,2025-03-01,1,0,0
1,FMR34103403,789103,2025-03-01,1,0,0
2,FMR32103602,789103,2025-03-01,1,0,0
3,FMR33103602,789103,2025-03-01,1,0,0
4,FMR33103401,789103,2025-03-01,1,0,0
...,...,...,...,...,...,...
13647,FJUN61301402,789301,2025-05-17,1,1,1
13648,FJUN61301602,789301,2025-05-17,1,1,1
13649,FMY530201602,789201,2025-05-17,1,1,1
13650,FMY530420203,789420,2025-05-17,1,1,1


In [5]:
fact_aggregate['order_placement_date'].min()

datetime.date(2025, 3, 1)

In [6]:
fact_aggregate['order_placement_date'].max()

datetime.date(2025, 5, 17)

# creation of dim date table

In [7]:
start_date = "2023-03-01"
end_date = "2025-05-31"


dates = pd.date_range(start=start_date, end=end_date)

dim_date = pd.DataFrame({
    "date": dates,
    "month" : dates.strftime("%Y-%m"),
    "year":dates.year,
    "week_of_year":dates.isocalendar().week

})

In [8]:
dim_date.head()

,date,month,year,week_of_year
2023-03-01,2023-03-01,2023-03,2023,9
2023-03-02,2023-03-02,2023-03,2023,9
2023-03-03,2023-03-03,2023-03,2023,9
2023-03-04,2023-03-04,2023-03,2023,9
2023-03-05,2023-03-05,2023-03,2023,9


# exchange date 

In [9]:
import numpy as np
exchange_df = pd.DataFrame({
    "date": pd.date_range("2023-03-01", "2025-05-31")
})

exchange_df["usd_to_inr"] = np.random.uniform(82, 86, len(exchange_df)).round(2)
    

In [10]:
exchange_df.head()

,date,usd_to_inr
0,2023-03-01,85.26
1,2023-03-02,82.53
2,2023-03-03,83.87
3,2023-03-04,84.75
4,2023-03-05,85.06


# Data cleaning and Transformation

In [11]:
dim_customers.dtypes

customer_id       int64
customer_name    object
city             object
currency         object
dtype: object

In [12]:
dim_customers.isnull().sum()

customer_id      0
customer_name    0
city             0
currency         0
dtype: int64

In [13]:
dim_customers.head()

,customer_id,customer_name,city,currency
0,789101,Costco Wholesale,"New Jersey, US",USD
1,789102,Vijay Stores,Ahmedabad,INR
2,789103,Vijay Stores,Vadodara,INR
3,789121,Coolblue,Ahmedabad,INR
4,789122,Coolblue,Vadodara,INR


In [14]:
dim_products.dtypes

product_name     object
product_id        int64
category         object
price_INR         int64
price_USD       float64
dtype: object

In [15]:
dim_products.isnull().sum()

product_name    0
product_id      0
category        0
price_INR       0
price_USD       0
dtype: int64

In [16]:
fact_aggregate.dtypes

order_id                object
customer_id              int64
order_placement_date    object
on_time                  int64
in_full                  int64
otif                     int64
dtype: object

In [17]:
fact_aggregate['order_placement_date']=pd.to_datetime(fact_aggregate['order_placement_date'])

In [18]:
fact_aggregate.isnull().sum()

order_id                0
customer_id             0
order_placement_date    0
on_time                 0
in_full                 0
otif                    0
dtype: int64

In [19]:
fact_aggregate.head()

,order_id,customer_id,order_placement_date,on_time,in_full,otif
0,FMR32103503,789103,2025-03-01,1,0,0
1,FMR34103403,789103,2025-03-01,1,0,0
2,FMR32103602,789103,2025-03-01,1,0,0
3,FMR33103602,789103,2025-03-01,1,0,0
4,FMR33103401,789103,2025-03-01,1,0,0


In [28]:
fact_aggregate.duplicated(subset=['order_id']).sum()

0

In [29]:
fact_aggregate=fact_aggregate.drop_duplicates(subset='order_id')

In [30]:
fact_order_line.dtypes

order_id                        object
order_placement_date    datetime64[ns]
customer_id                      int64
product_id                       int64
order_qty                        int64
agreed_delivery_date    datetime64[ns]
actual_delivery_date    datetime64[ns]
delivery_qty                     int64
In Full                          int64
On Time                          int64
On Time In Full                  int64
dtype: object

In [31]:
fact_order_line['agreed_delivery_date']=pd.to_datetime(fact_order_line['agreed_delivery_date'])
fact_order_line['order_placement_date']=pd.to_datetime(fact_order_line['order_placement_date'])
fact_order_line['actual_delivery_date']=pd.to_datetime(fact_order_line['actual_delivery_date'])

In [27]:
fact_order_line.duplicated(subset=['order_id','product_id']).sum()

0

In [32]:
fact_order_line=fact_order_line.drop_duplicates(subset=['order_id','product_id'])

In [33]:
#merge process
flatten_df=(fact_order_line
            .merge(fact_aggregate,on=['order_id','customer_id','order_placement_date'],how='left',suffixes=('_line', '_agg'))
            .merge(dim_products,on=['product_id'],how='left')
            .merge(dim_customers,on=['customer_id'],how='left')
            .merge(exchange_df,left_on=['order_placement_date'],right_on=['date'],how='left')
           )
                                   


In [34]:
# duplicate check
flatten_df.loc[flatten_df.duplicated(subset=['order_id','product_id'])]

,order_id,order_placement_date,customer_id,product_id,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty,In Full,On Time,...,otif,product_name,category,price_INR,price_USD,customer_name,city,currency,date,usd_to_inr


In [35]:
flatten_df.columns

Index(['order_id', 'order_placement_date', 'customer_id', 'product_id',
       'order_qty', 'agreed_delivery_date', 'actual_delivery_date',
       'delivery_qty', 'In Full', 'On Time', 'On Time In Full', 'on_time',
       'in_full', 'otif', 'product_name', 'category', 'price_INR', 'price_USD',
       'customer_name', 'city', 'currency', 'date', 'usd_to_inr'],
      dtype='object')

In [36]:
# currency conversion
flatten_df['Amount_INR']=np.where(flatten_df['currency']=='USD',flatten_df['price_USD']*flatten_df['usd_to_inr']*flatten_df['delivery_qty'],flatten_df['price_INR'])

In [37]:
flatten_df.rename(columns={'In Full':'Item_In_Full','On Time':'Item_On_Time',
                           'On Time In Full':'Item_On_Time_In_Full','on_time':'ord_on_time','in_full':'ord_in_full','otif':'ord_otif'},inplace=True)

In [38]:
flatten_df.columns

Index(['order_id', 'order_placement_date', 'customer_id', 'product_id',
       'order_qty', 'agreed_delivery_date', 'actual_delivery_date',
       'delivery_qty', 'Item_In_Full', 'Item_On_Time', 'Item_On_Time_In_Full',
       'ord_on_time', 'ord_in_full', 'ord_otif', 'product_name', 'category',
       'price_INR', 'price_USD', 'customer_name', 'city', 'currency', 'date',
       'usd_to_inr', 'Amount_INR'],
      dtype='object')

In [39]:
temp_tbl=flatten_df[['order_id', 'order_placement_date', 'customer_id','customer_name', 'city','product_id',
                     'product_name', 'category','order_qty', 'agreed_delivery_date', 'actual_delivery_date','delivery_qty', 
                     'Item_In_Full', 'Item_On_Time', 'Item_On_Time_In_Full','ord_on_time', 'ord_in_full', 'ord_otif', 'Amount_INR']]

In [40]:
temp_tbl

,order_id,order_placement_date,customer_id,customer_name,city,product_id,product_name,category,order_qty,agreed_delivery_date,actual_delivery_date,delivery_qty,Item_In_Full,Item_On_Time,Item_On_Time_In_Full,ord_on_time,ord_in_full,ord_otif,Amount_INR
0,FMR34203601,2025-03-01,789203,Rel Fresh,Vadodara,25891601,AM Tea 500,beverages,110,2025-03-04,2025-03-04,110,1,1,1,1,1,1,225.00
1,FMR32320302,2025-03-01,789320,Whole Foods Market,"New Jersey, US",25891203,AM Butter 500,Dairy,347,2025-03-02,2025-03-02,347,1,1,1,1,1,1,256616.91
2,FMR33320501,2025-03-01,789320,Whole Foods Market,"New Jersey, US",25891203,AM Butter 500,Dairy,187,2025-03-03,2025-03-03,150,0,1,0,1,0,0,110929.50
3,FMR34220601,2025-03-01,789220,ShopRite,"New Jersey, US",25891203,AM Butter 500,Dairy,235,2025-03-04,2025-03-04,235,1,1,1,1,1,1,173789.55
4,FMR33703603,2025-03-01,789703,Sorefoz Mart,Vadodara,25891203,AM Butter 500,Dairy,176,2025-03-03,2025-03-03,176,1,1,1,1,1,1,300.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24525,FMY531201301,2025-05-17,789201,Rel Fresh,"New Jersey, US",25891301,AM Ghee 250,Dairy,40,2025-05-31,2025-06-02,40,1,0,0,0,0,0,20308.80
24526,FMY531301602,2025-05-17,789301,Foodtown,"New Jersey, US",25891301,AM Ghee 250,Dairy,41,2025-05-31,2025-05-31,41,1,1,1,1,1,1,20816.52
24527,FJUN61401301,2025-05-17,789401,Wegmans,"New Jersey, US",25891301,AM Ghee 250,Dairy,45,2025-06-01,2025-06-04,45,1,0,0,0,1,0,22847.40
24528,FMY530720502,2025-05-17,789720,Colonial Farms,"New Jersey, US",25891102,AM Milk 250,Dairy,457,2025-05-30,2025-05-30,366,0,1,0,1,0,0,61941.84


# exporting flatten table as csv

In [57]:
temp_tbl.to_csv('summary.csv')

#TESTING VALUES

In [42]:
Total_Orders = temp_tbl['order_id'].nunique()
Total_Orders

13652

In [43]:
# Line fill rate delivery qty >= ordered_qty
Total_Line_fill_rate = temp_tbl.loc[temp_tbl['delivery_qty']>=temp_tbl['order_qty']].shape[0]
Total_Line_fill_rate

16177

In [44]:
Total_Line_fill_rate_pct = round(((Total_Line_fill_rate/temp_tbl.shape[0])*100),2)
Total_Line_fill_rate_pct

65.95

In [45]:
# Volume Fill Rate 
Volume_fill_rate_pct = round((temp_tbl['delivery_qty'].sum() / temp_tbl['order_qty'].sum()*100),2)
Volume_fill_rate_pct

96.6

In [46]:
# Ontime Delivery 
Ontime_Delivery_pct=round(temp_tbl.loc[(temp_tbl['ord_on_time']==1),['order_id']].nunique()/Total_Orders*100,2)
Ontime_Delivery_pct

order_id    59.22
dtype: float64

In [56]:
InTime_Delivery_pct=round(temp_tbl.loc[temp_tbl['ord_in_full']==1,['order_id']].nunique()/Total_Orders*100,2)
InTime_Delivery_pct

order_id    52.62
dtype: float64

In [55]:
InFull_On_Time_Delivery_pct=round(temp_tbl.loc[temp_tbl['ord_otif']==1,['order_id']].nunique()/Total_Orders*100,2)
InTime_Delivery_pct

order_id    28.74
dtype: float64

In [115]:
!streamlit run app.py

^C
